In [379]:
import paddlenlp
from datasets import load_dataset

train_dataset_path = "data/data174918/train.json"
dev_dataset_path = "data/data174918/dev.json"
test_dataset_path = "data/data174918/test.json"
dataset = load_dataset("json",
                       data_files={
                           "train":train_dataset_path,
                            "dev":dev_dataset_path,
                            "test":test_dataset_path})
dataset["test"][:3]

{'text': ['共 同 创 造 美 好 的 新 世 纪 — — 二 ○ ○ 一 年 新 年 贺 词',
  '( 二 ○ ○ ○ 年 十 二 月 三 十 一 日 ) ( 附 图 片 1 张 )',
  '女 士 们 , 先 生 们 , 同 志 们 , 朋 友 们 :'],
 'label': ['B E B E B E S S B E B E B M M M E B E B E',
  'S B M M M E B M E B M M E S S S B E S S S',
  'B E S S B E S S B E S S B E S S']}

In [380]:
from paddlenlp.transformers import BertTokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-chinese')
inputs = tokenizer(text="中文分词是一项重要的自然语言处理领域任务")
inputs

[2026-02-23 22:19:22,671] [    INFO] - Already cached /home/ricaedo/.paddlenlp/models/bert-base-chinese/bert-base-chinese-vocab.txt
[2026-02-23 22:19:22,676] [    INFO] - tokenizer config file saved in /home/ricaedo/.paddlenlp/models/bert-base-chinese/tokenizer_config.json
[2026-02-23 22:19:22,676] [    INFO] - Special tokens file saved in /home/ricaedo/.paddlenlp/models/bert-base-chinese/special_tokens_map.json


{'input_ids': [101, 704, 3152, 1146, 6404, 3221, 671, 7555, 7028, 6206, 4638, 5632, 4197, 6427, 6241, 1905, 4415, 7566, 1818, 818, 1218, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}

In [381]:
tokens = tokenizer.tokenize('中文分词是一项重要的自然语言处理领域任务')
print(tokens,'\n',len(tokens))
ids = tokenizer.convert_tokens_to_ids(tokens)
print(ids,'\n',len(ids))
tokens_ = tokenizer.convert_ids_to_tokens(ids)
print(tokens_,'\n',len(tokens_))
strings = tokenizer.convert_tokens_to_string(tokens)
print(strings)

['中', '文', '分', '词', '是', '一', '项', '重', '要', '的', '自', '然', '语', '言', '处', '理', '领', '域', '任', '务'] 
 20
[704, 3152, 1146, 6404, 3221, 671, 7555, 7028, 6206, 4638, 5632, 4197, 6427, 6241, 1905, 4415, 7566, 1818, 818, 1218] 
 20
['中', '文', '分', '词', '是', '一', '项', '重', '要', '的', '自', '然', '语', '言', '处', '理', '领', '域', '任', '务'] 
 20
中 文 分 词 是 一 项 重 要 的 自 然 语 言 处 理 领 域 任 务


In [382]:
from paddlenlp.transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")
print(tokenizer.__class__)
inputs = tokenizer(text="中文分词是一项重要的自然语言处理领域任务")
inputs

[2026-02-23 22:19:22,696] [    INFO] - We are using (<class 'paddlenlp.transformers.bert.tokenizer.BertTokenizer'>, False) to load 'bert-base-chinese'.
[2026-02-23 22:19:22,697] [    INFO] - Already cached /home/ricaedo/.paddlenlp/models/bert-base-chinese/bert-base-chinese-vocab.txt
[2026-02-23 22:19:22,700] [    INFO] - tokenizer config file saved in /home/ricaedo/.paddlenlp/models/bert-base-chinese/tokenizer_config.json
[2026-02-23 22:19:22,701] [    INFO] - Special tokens file saved in /home/ricaedo/.paddlenlp/models/bert-base-chinese/special_tokens_map.json


<class 'paddlenlp.transformers.bert.tokenizer.BertTokenizer'>


{'input_ids': [101, 704, 3152, 1146, 6404, 3221, 671, 7555, 7028, 6206, 4638, 5632, 4197, 6427, 6241, 1905, 4415, 7566, 1818, 818, 1218, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}

In [383]:
def convert_example_to_feature(example,tokenizer,label2id,max_seq_len=512,is_infer=False):
    text = example['text'].strip().split(" ")
    encoded_inputs = tokenizer(text=text,
                                max_seq_len=max_seq_len,
                                is_split_into_words="token",
                                return_length=True)
    encoded_inputs['label'] = []
    try:
        if not is_infer:
            ids = []
            offset = 0
            for id,word_ids in enumerate(encoded_inputs['input_ids']):
                token = tokenizer.convert_ids_to_tokens([word_ids])[-1]
                if token in ["##1","##2","##3","##4","##5","##6","##7","##8","##9","##0"]:
                    offset+=1
                    ids.append(f_ids)
                elif token in ["[CLS]","[SEP]","[UNK]"]:
                    offset+=1
                    ids.append(label2id["O"])
                else:
                    f_ids = label2id[example['label'].strip().split(" ")[id - offset]]
                    ids.append(f_ids) 
            ids = ids[:max_seq_len]
            encoded_inputs["label"] = ids
            assert len(encoded_inputs["label"]) == len(encoded_inputs["input_ids"]) ,f"{tokenizer.convert_ids_to_tokens(encoded_inputs['input_ids'])},\n,{len(encoded_inputs['label'])},{len(encoded_inputs['input_ids'])}"
    except Exception as e:

        raise e
    finally:
        return encoded_inputs
model_name = "bert-base-chinese"
label2id = {"O":0, "B":1, "M":2, "E":3, "S":4}
tokenizer = AutoTokenizer.from_pretrained(model_name)

[2026-02-23 22:19:22,708] [    INFO] - We are using (<class 'paddlenlp.transformers.bert.tokenizer.BertTokenizer'>, False) to load 'bert-base-chinese'.
[2026-02-23 22:19:22,708] [    INFO] - Already cached /home/ricaedo/.paddlenlp/models/bert-base-chinese/bert-base-chinese-vocab.txt
[2026-02-23 22:19:22,713] [    INFO] - tokenizer config file saved in /home/ricaedo/.paddlenlp/models/bert-base-chinese/tokenizer_config.json
[2026-02-23 22:19:22,713] [    INFO] - Special tokens file saved in /home/ricaedo/.paddlenlp/models/bert-base-chinese/special_tokens_map.json


In [384]:
example = {"text":"钱 其 琛 访 问 德 班", "label":"S B E B E B E"}
features = convert_example_to_feature(example, tokenizer, label2id, max_seq_len=512, is_infer=False)
features

/home/ricaedo/下载/yes/envs/paddle/lib/python3.10/site-packages/paddlenlp/transformers/tokenizer_utils_base.py:2353: FutureWarning: The `max_seq_len` argument is deprecated and will be removed in a future version, please use `max_length` instead.
  warnings.warn(
/home/ricaedo/下载/yes/envs/paddle/lib/python3.10/site-packages/paddlenlp/transformers/tokenizer_utils_base.py:1925: UserWarning: Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
  warnings.warn(


{'input_ids': [101, 7178, 1071, 4422, 6393, 7309, 2548, 4408, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0], 'length': 9, 'seq_len': 9, 'label': [0, 4, 1, 3, 1, 3, 1, 3, 0]}

In [385]:
from functools import partial
trans_fn = partial(convert_example_to_feature,tokenizer=tokenizer,label2id=label2id,max_seq_len=512)

columns = ["text", "label"]
train_dataset = dataset["train"].map(trans_fn, batched=False, remove_columns=columns)
dev_dataset = dataset["dev"].map(trans_fn, batched=False, remove_columns=columns)
test_dataset = dataset["test"].map(trans_fn, batched=False, remove_columns=columns)

print("train_dataset:", len(train_dataset))
print("dev_dataset:", len(dev_dataset))
print("test_dataset:", len(test_dataset))


Map: 100%|██████████| 1944/1944 [00:02<00:00, 947.32 examples/s] 

train_dataset: 18031
dev_dataset: 1000
test_dataset: 1944
